In [ ]:
import scipy.stats as stats
from scipy.optimize import brentq

def eq_for_c(c, eps, var):
    left_side = stats.norm.pdf((stats.norm.ppf(1-c*eps)))/(c*eps)
    return left_side - var

def find_c(eps, var):
    c_sol = brentq(eq_for_c, 1, 1 / eps, args=(eps, var), maxiter=1000, xtol=1e-15)
    return c_sol

eps = 0.01
var = stats.norm.ppf(1 - eps)

c = find_c(eps, var)
print(f"c = {c}")

for _ in range(0, 15):
    var = var = stats.norm.ppf(1 - eps)
    c = find_c(eps, var)

    print(f"{eps:<10.1e} | {c:<8.5f}")
    eps /= 10


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import scipy.stats as stats
from scipy.integrate import quad
from scipy.optimize import brentq

def norm_isf(p):
    return stats.norm.isf(p)

def eq_for_c(c, eps, var):
    def integrand(t):
        p = c * eps * t
        return norm_isf(p)
    integral, _ = quad(integrand, 0, 1, epsabs=1e-14, epsrel=1e-14, limit=1000)
    return integral - var

def find_c(eps, var):
    c_min = 1.0
    c_max = 1 / eps
    return brentq(eq_for_c, c_min, c_max,
                  args=(eps, var),
                  maxiter=10000, xtol=1e-14)

num_points_linear = 40
num_points_log = 50
total_points = num_points_linear + num_points_log

eps_linear = np.linspace(1e-1, 1e-5, 40, endpoint=False)

eps_log = np.logspace(-2.585, -20, 50)

eps_values_combined = np.concatenate((eps_linear, eps_log))

x_values = 1.0 - eps_values_combined

c_results = []

print(f"Calculating c for {total_points} combined values of eps:")

count_nan = 0
for i, eps in enumerate(eps_values_combined):

    var = norm_isf(eps)
    c = find_c(eps, var)

    c_results.append(c)

    if (i + 1) % (total_points // 10) == 0:
        print(f"Processed {i+1}/{total_points} points...")

print(f"Calculation finished. {count_nan} points resulted in NaN for c.")

x_values_plot = np.array(x_values)
c_results_plot = np.array(c_results)

valid_indices = ~np.isnan(c_results_plot)
x_plot = x_values_plot[valid_indices]
c_plot = c_results_plot[valid_indices]


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))

if len(x_plot) > 0:

    plt.plot(x_plot, c_plot, marker='', linestyle='-', markersize=4, label=r'$c(\varepsilon)$')
else:
    print("Warning: No valid points to plot.")

euler_e = math.e
plt.axhline(y=euler_e, color='red', linestyle='-', linewidth=1.5,
            label=fr'$y = e \approx {euler_e:.5f}$')

plt.rc('text', usetex=False)
plt.xlabel(r'$1 - \varepsilon$', fontsize=14)
plt.ylabel(r'$\Pi_\varepsilon(X)$', fontsize=14)

plt.title(r'', fontsize=16)

if len(c_plot) > 0:

     min_c_val = np.nanmin(c_plot)
     max_c_val = np.nanmax(c_plot)

     if np.isfinite(min_c_val) and np.isfinite(max_c_val) and np.isfinite(euler_e):
         plot_min_y = min(min_c_val * 0.995, euler_e * 0.99)
         plot_max_y = max(max_c_val * 1.005, euler_e * 1.01)
         y_range = plot_max_y - plot_min_y

         if np.isfinite(y_range) and y_range > 1e-9:
             plt.ylim(plot_min_y - 0.01*y_range, plot_max_y + 0.01*y_range)
         elif np.isfinite(plot_min_y) and np.isfinite(plot_max_y):
              plt.ylim(plot_min_y - 0.1, plot_max_y + 0.1)

plt.grid(True)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import scipy.stats as stats
from scipy.integrate import quad
from scipy.optimize import brentq

def norm_isf(p):
    return stats.norm.isf(p)

def eq_for_c(c, eps, var):
    def integrand(t):
        p = c * eps * t
        return norm_isf(p)
    integral, _ = quad(integrand, 0, 1, epsabs=1e-14, epsrel=1e-14, limit=1000)
    return integral - var

def find_c(eps, var):
    c_min = 1.0
    c_max = 1 / eps
    return brentq(eq_for_c, c_min, c_max,
                  args=(eps, var),
                  maxiter=10000, xtol=1e-14)

eps = 1e-308
var = norm_isf(eps)
c = find_c(eps, var)

print(c)
